In [4]:
import requests
import pandas as pd
import numpy as np


In [2]:
China_EV = pd.read_csv("/home/wang277y/Data/China_EV.csv")

In [3]:
China_EV.info

<bound method DataFrame.info of     3 month rolling avg (thousand)  US total vehicle sales  China EV sales  \
0                       2025-05-01                 1544.20         1179.40   
1                       2025-04-01                 1458.98         1048.95   
2                       2025-03-01                 1339.77          966.90   
3                       2025-02-01                 1309.60         1086.86   
4                       2025-01-01                 1364.18         1289.10   
..                             ...                     ...             ...   
120                     2015-05-01                 1576.83            8.85   
121                     2015-04-01                 1448.86            7.08   
122                     2015-03-01                 1348.20            5.97   
123                     2015-02-01                 1334.25            4.51   
124                     2015-01-01                 1347.03            4.47   

    China's EV sales as a share

In [22]:
# keep only China EV sales and percentage
China_EV = China_EV[[
    "3 month rolling avg (thousand)",
    "China EV sales",
    "China's EV sales as a share of total sales"
]].copy()

In [23]:
# rename columns
China_EV = China_EV.rename(columns={
    "3 month rolling avg (thousand)": "month",
    "China EV sales": "china_ev_sales",
    "China's EV sales as a share of total sales": "china_ev_share"
})

In [24]:
# EV sales -> numeric

China_EV["china_ev_share"] = (
    China_EV["china_ev_share"]
    .astype(str)
    .str.replace("%", "")
    .str.strip()
)

China_EV["china_ev_share"] = pd.to_numeric(China_EV["china_ev_share"], errors="coerce")



In [25]:
# remove missing values
China_EV = China_EV.dropna()



In [26]:
clean.head()

,month,china_ev_sales_thousand,china_ev_share_pct
0,2025-05-01,1179.40,52.5
1,2025-04-01,1048.95,51.6
2,2025-03-01,966.90,46.9
3,2025-02-01,1086.86,46.4
4,2025-01-01,1289.10,42.2


In [17]:
import sqlite3 as sq

In [18]:
connection = sq.connect("china_ev.db")

In [27]:
# save to SQL table
China_EV.to_sql(
    "china_ev_sales",
    connection,
    if_exists="replace",
    index=False
)

125

In [28]:
# Check the cleaned data
query = """
SELECT *
FROM china_ev_sales
LIMIT 20;
"""

pd.read_sql(query, connection)

,month,china_ev_sales,china_ev_share
0,2025-05-01,1179.40,52.5
1,2025-04-01,1048.95,51.6
2,2025-03-01,966.90,46.9
3,2025-02-01,1086.86,46.4
4,2025-01-01,1289.10,42.2
5,2024-12-01,1446.09,48.6
6,2024-11-01,1351.83,48.3
7,2024-10-01,1220.12,49.8
8,2024-09-01,1080.78,48.9
9,2024-08-01,1003.20,48.4


In [31]:
# Total EV sales 
query = """
SELECT
SUM(china_ev_sales) AS total_ev_sales
FROM china_ev_sales;
"""
pd.read_sql(query, connection)

,total_ev_sales
0,39563.64


This shows us the total EV sales in the data set by summing the ev sales. 

In [33]:
# Average EV market share 
query = """
SELECT
AVG(china_ev_share) AS avg_ev_share
FROM china_ev_sales;
"""
pd.read_sql(query, connection)

,avg_ev_share
0,15.3352


This query shows us the average ev share from the dataset by using the AVG function. It allows us to see a snapshot at how much China has been working towards increased electric vehicle usage. 

In [34]:
# Month with highest EV sales 
query = """
SELECT
month,
china_ev_sales
FROM china_ev_sales
ORDER BY china_ev_sales DESC
LIMIT 1;
"""
pd.read_sql(query, connection)

,month,china_ev_sales
0,2024-12-01,1446.09


In [35]:
# EV sales trend over time
query = """
SELECT
month,
china_ev_sales
FROM china_ev_sales
ORDER BY month;
"""
pd.read_sql(query, connection)

,month,china_ev_sales
0,2015-01-01,4.47
1,2015-02-01,4.51
2,2015-03-01,5.97
3,2015-04-01,7.08
4,2015-05-01,8.85
...,...,...
120,2025-01-01,1289.10
121,2025-02-01,1086.86
122,2025-03-01,966.90
123,2025-04-01,1048.95


#### How have we seen electric vehicle sales increase overtime?
We have seen a great increase in the sale of electric vehicles in the past decade. The query selects from the sales column and orders by date for us to view this information.

In [61]:
# Use Case when to convert EV sales numbers into Low, Medium, and High levels.
query = """
SELECT
  month,
  china_ev_sales,

  CASE
    WHEN china_ev_sales < 500 THEN 'Low EV Sales'
    WHEN china_ev_sales BETWEEN 500 AND 1000 THEN 'Medium EV Sales'
    ELSE 'High EV Sales'
  END AS ev_sales_level

FROM china_ev_sales
ORDER BY month;
"""

pd.read_sql(query, connection)

,month,china_ev_sales,ev_sales_level
0,2015-01-01,4.47,Low EV Sales
1,2015-02-01,4.51,Low EV Sales
2,2015-03-01,5.97,Low EV Sales
3,2015-04-01,7.08,Low EV Sales
4,2015-05-01,8.85,Low EV Sales
...,...,...,...
120,2025-01-01,1289.10,High EV Sales
121,2025-02-01,1086.86,High EV Sales
122,2025-03-01,966.90,Medium EV Sales
123,2025-04-01,1048.95,High EV Sales


 We can see that The sales of EV in chinese market is increasingly to grow from 2015-2025

In [36]:
import requests
import pandas as pd
import numpy as np

# Beijing coordinates
LAT, LON = 39.9042, 116.4074

def fetch_open_meteo_air_quality(lat, lon, start_date, end_date):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ["pm2_5", "pm10", "ozone", "nitrogen_dioxide", "carbon_monoxide", "sulphur_dioxide"],
        "start_date": start_date,
        "end_date": end_date,
        "timezone": "Asia/Shanghai"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df

aq_hourly = fetch_open_meteo_air_quality(LAT, LON, "2024-02-01", "2026-02-01")
aq_hourly.head()

,time,pm2_5,pm10,ozone,nitrogen_dioxide,carbon_monoxide,sulphur_dioxide
0,2024-02-01 00:00:00,121.1,174.2,0.0,56.8,1875.0,116.9
1,2024-02-01 01:00:00,93.4,134.6,0.0,58.4,1447.0,84.9
2,2024-02-01 02:00:00,76.6,110.4,0.0,59.5,1117.0,61.1
3,2024-02-01 03:00:00,68.9,99.2,0.0,59.6,979.0,52.5
4,2024-02-01 04:00:00,63.3,91.0,0.0,59.2,938.0,52.0


In [37]:
# save hourly air quality into SQLite
aq_hourly.to_sql("beijing_aq_hourly", connection, if_exists="replace", index=False)

17568

In [38]:
pd.read_sql("SELECT * FROM beijing_aq_hourly LIMIT 5;", connection)

,time,pm2_5,pm10,ozone,nitrogen_dioxide,carbon_monoxide,sulphur_dioxide
0,2024-02-01 00:00:00,121.1,174.2,0.0,56.8,1875.0,116.9
1,2024-02-01 01:00:00,93.4,134.6,0.0,58.4,1447.0,84.9
2,2024-02-01 02:00:00,76.6,110.4,0.0,59.5,1117.0,61.1
3,2024-02-01 03:00:00,68.9,99.2,0.0,59.6,979.0,52.5
4,2024-02-01 04:00:00,63.3,91.0,0.0,59.2,938.0,52.0


In [45]:
# drop + rebuild PM2.5-only monthly table
connection.execute("DROP TABLE IF EXISTS beijing_pm25_monthly;")

connection.execute("""
CREATE TABLE beijing_pm25_monthly AS
SELECT
  strftime('%Y-%m', datetime(time)) AS month,
  AVG(pm2_5) AS avg_pm2_5
FROM beijing_aq_hourly
WHERE pm2_5 IS NOT NULL
GROUP BY strftime('%Y-%m', datetime(time));
""")

pd.read_sql("SELECT * FROM beijing_pm25_monthly ORDER BY month LIMIT 10;", connection)

,month,avg_pm2_5
0,2024-02,104.984052
1,2024-03,67.950269
2,2024-04,55.334028
3,2024-05,48.524731
4,2024-06,52.680833
5,2024-07,70.335484
6,2024-08,78.549597
7,2024-09,68.420000
8,2024-10,88.001210
9,2024-11,115.135139


In [50]:
# Clean the dataset
connection.execute("DROP VIEW IF EXISTS china_ev_month;")

connection.execute("""
CREATE VIEW china_ev_month AS
SELECT
  substr(month, 1, 7) AS month,
  china_ev_sales
FROM china_ev_sales;
""")

pd.read_sql("SELECT * FROM china_ev_month ORDER BY month LIMIT 10;", connection)

,month,china_ev_sales
0,2015-01,4.47
1,2015-02,4.51
2,2015-03,5.97
3,2015-04,7.08
4,2015-05,8.85
5,2015-06,9.76
6,2015-07,10.79
7,2015-08,11.17
8,2015-09,12.78
9,2015-10,16.08


In [53]:
# Use innerjoin to show rows that only have matching values in both tables to be used for further analysis 
query = """
SELECT
  e.month,
  e.china_ev_sales,
  p.avg_pm2_5
FROM china_ev_month e
INNER JOIN beijing_pm25_monthly p
  ON e.month = p.month
ORDER BY e.month;
"""
ev_pm = pd.read_sql(query, connection)
ev_pm.head(12)

,month,china_ev_sales,avg_pm2_5
0,2024-02,759.30,104.984052
1,2024-03,660.64,67.950269
2,2024-04,696.36,55.334028
3,2024-05,849.52,48.524731
4,2024-06,906.29,52.680833
5,2024-07,955.24,70.335484
6,2024-08,1003.20,78.549597
7,2024-09,1080.78,68.420000
8,2024-10,1220.12,88.001210
9,2024-11,1351.83,115.135139


In [55]:
#save ev_pm to sql
ev_pm.to_sql("ev_pm", connection, if_exists="replace", index=False)

16

In [56]:
# Month-to-month change (lag window function) to show whether EV growth corresponds to PM2.5 decline.
query = """
SELECT
  month,
  china_ev_sales,
  avg_pm2_5,

  china_ev_sales - LAG(china_ev_sales)
      OVER (ORDER BY month) AS ev_sales_change,

  avg_pm2_5 - LAG(avg_pm2_5)
      OVER (ORDER BY month) AS pm25_change

FROM ev_pm
ORDER BY month;
"""

pd.read_sql(query, connection)

,month,china_ev_sales,avg_pm2_5,ev_sales_change,pm25_change
0,2024-02,759.30,104.984052,NaN,NaN
1,2024-03,660.64,67.950269,-98.66,-37.033783
2,2024-04,696.36,55.334028,35.72,-12.616241
3,2024-05,849.52,48.524731,153.16,-6.809297
4,2024-06,906.29,52.680833,56.77,4.156102
5,2024-07,955.24,70.335484,48.95,17.654651
6,2024-08,1003.20,78.549597,47.96,8.214113
7,2024-09,1080.78,68.420000,77.58,-10.129597
8,2024-10,1220.12,88.001210,139.34,19.581210
9,2024-11,1351.83,115.135139,131.71,27.133929


#### Does EV growth corresponds to PM2.5 decline?
The month-to-month change analysis shows whether increases in EV adoption correspond to decreases in PM2.5 levels. 
If EV growth coincides with declining PM2.5 concentrations, it suggests a potential environmental benefit from electrification. 
However, if both increase simultaneously, other pollution sources such as industry or heating demand may dominate air quality outcomes. Unfortunately, results are inconclusive from this query. 

In [58]:
# We try to find out whether high EV months have lower PM2.5.
query = """
SELECT
  month,
  china_ev_sales,
  avg_pm2_5,

  RANK() OVER (ORDER BY china_ev_sales DESC) AS ev_rank,
  RANK() OVER (ORDER BY avg_pm2_5 ASC) AS clean_air_rank

FROM ev_pm
ORDER BY month;
"""

pd.read_sql(query, connection)

,month,china_ev_sales,avg_pm2_5,ev_rank,clean_air_rank
0,2024-02,759.30,104.984052,14,13
1,2024-03,660.64,67.950269,16,6
2,2024-04,696.36,55.334028,15,5
3,2024-05,849.52,48.524731,13,2
4,2024-06,906.29,52.680833,12,3
5,2024-07,955.24,70.335484,11,8
6,2024-08,1003.20,78.549597,9,9
7,2024-09,1080.78,68.420000,7,7
8,2024-10,1220.12,88.001210,4,12
9,2024-11,1351.83,115.135139,2,15


#### Do high EV sale months correspond with a drop in pm25 pollution?
This query compares these two factors using a rank system for sales and pollution. It selects on these columns, ranks them, and then is ordered by month. Again, results are inconclusive. 

In [59]:
ev_pm[["china_ev_sales","avg_pm2_5"]].corr()

,china_ev_sales,avg_pm2_5
china_ev_sales,1.000000,0.569738
avg_pm2_5,0.569738,1.000000


The correlation analysis shows a moderate positive relationship (0.57) between EV sales and PM2.5 concentrations. 
This indicates that periods with higher EV adoption also coincided with higher pollution levels in the dataset. 
This does not imply that EVs worsen air quality. It could due to other factors such as industrial activity, 
energy generation sources, and seasonal effects likely have a stronger influence on PM2.5 levels in the short term.

In [1]:
import sqlite3 as sq

In [2]:
import requests
import pandas as pd
import numpy as np

# Beijing coordinates
LAT, LON = 39.9042, 116.4074

def fetch_open_meteo_air_quality(lat, lon, start_date, end_date):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ["pm2_5", "pm10", "ozone", "nitrogen_dioxide", "carbon_monoxide", "sulphur_dioxide"],
        "start_date": start_date,
        "end_date": end_date,
        "timezone": "Asia/Shanghai"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    df = pd.DataFrame(data["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df

aq_hourly = fetch_open_meteo_air_quality(LAT, LON, "2024-02-01", "2026-02-01")
aq_hourly.head()

,time,pm2_5,pm10,ozone,nitrogen_dioxide,carbon_monoxide,sulphur_dioxide
0,2024-02-01 00:00:00,121.1,174.2,0.0,56.8,1875.0,116.9
1,2024-02-01 01:00:00,93.4,134.6,0.0,58.4,1447.0,84.9
2,2024-02-01 02:00:00,76.6,110.4,0.0,59.5,1117.0,61.1
3,2024-02-01 03:00:00,68.9,99.2,0.0,59.6,979.0,52.5
4,2024-02-01 04:00:00,63.3,91.0,0.0,59.2,938.0,52.0


In [4]:
connection = sq.connect("aq_hourly.db")
aq_hourly.to_sql("beijing_aq_hourly", connection, if_exists="replace", index=False)

17568

In [13]:
connection.execute("DROP TABLE IF EXISTS beijing_pm25_monthly;")

# drop + rebuild PM2.5-only monthly table
connection.execute("""
CREATE TABLE beijing_pm25_monthly AS
SELECT
  strftime('%Y-%m', datetime(time)) AS month,
  AVG(pm2_5) AS avg_pm2_5
FROM beijing_aq_hourly
WHERE pm2_5 IS NOT NULL
GROUP BY strftime('%Y-%m', datetime(time));
""")

pd.read_sql("SELECT * FROM beijing_pm25_monthly ORDER BY month LIMIT 10;", connection)

,month,avg_pm2_5
0,2024-02,104.984052
1,2024-03,67.950269
2,2024-04,55.334028
3,2024-05,48.524731
4,2024-06,52.680833
5,2024-07,70.335484
6,2024-08,78.549597
7,2024-09,68.420000
8,2024-10,88.001210
9,2024-11,115.135139


In [56]:
#create cursor 

cursor = connection.cursor()

#inserting weather conditions csv and making connection 
Beijing_Weather_df = pd.read_csv('/home/makiyahm/Midterm/POWER_Point_Daily_20240201_20260201_.csv')

Beijing_Weather_df.to_sql("Beijing_Weather", connection, if_exists="replace", index=False)

732

In [57]:
#creating a date column in the table to complete an inner join later 

cursor = connection.cursor()

cursor.execute("ALTER TABLE Beijing_Weather ADD COLUMN date TEXT")


In [58]:
cursor.execute("""
UPDATE Beijing_Weather
SET date = date(YEAR || '-01-01', '+' || (DOY - 1) || ' days')
""")

connection.commit()

pd.read_sql("SELECT DOY, date FROM Beijing_Weather LIMIT 10", connection)

,DOY,date
0,32,2024-02-01
1,33,2024-02-02
2,34,2024-02-03
3,35,2024-02-04
4,36,2024-02-05
5,37,2024-02-06
6,38,2024-02-07
7,39,2024-02-08
8,40,2024-02-09
9,41,2024-02-10


In [59]:
#checking the table names because of a previous error. Beijing Weather Conditions was not working
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", connection)

,name
0,beijing_aq_hourly
1,Beijing Weather Conditions
2,beijing_pm25_monthly
3,Beijing_Weather


In [18]:
#checking the weather conditions table 
pd.read_sql("SELECT * FROM Beijing_Weather LIMIT 5;", connection)

,YEAR,DOY,T2M,RH2M,PRECTOTCORR,WS2M,PS,date
0,2024,32,-7.51,49.30,0.00,0.97,100.97,2024-02-01
1,2024,33,-6.63,59.39,0.00,1.02,100.62,2024-02-02
2,2024,34,-3.66,55.85,0.00,1.07,100.72,2024-02-03
3,2024,35,-4.91,64.77,0.24,1.10,100.49,2024-02-04
4,2024,36,-2.74,65.55,0.42,1.15,100.22,2024-02-05


In [30]:
#attempting to change the column names using the AS function. Did not work for an actual replacement  

query = """
SELECT 
    T2M AS Temperature,
    RH2M AS Humidity,
    PRECTOTCORR AS Precipitation,
    WS2M AS wind_speed,
    PS AS Pressure
FROM Beijing_Weather
"""

df = pd.read_sql(query, connection)
df.head()

,Temperature,Humidity,Precipitation,wind_speed,Pressure
0,-7.51,49.30,0.00,0.97,100.97
1,-6.63,59.39,0.00,1.02,100.62
2,-3.66,55.85,0.00,1.07,100.72
3,-4.91,64.77,0.24,1.10,100.49
4,-2.74,65.55,0.42,1.15,100.22


In [23]:
#deleteing the rows that are null, we already know this from our previous work on the data set

cursor.execute("""
DELETE FROM Beijing_Weather
WHERE T2M = -999
   OR RH2M = -999
   OR PRECTOTCORR = -999
   OR WS2M = -999
   OR PS = -999
""")

connection.commit()

In [25]:
#the year column is unnecessary in this table now that we have the date column 

cursor.execute("""
ALTER TABLE Beijing_Weather
DROP COLUMN YEAR
""")

connection.commit()

In [52]:
#checking the column names of the table 
pd.read_sql("PRAGMA table_info(Beijing_Weather);", connection)

,cid,name,type,notnull,dflt_value,pk
0,0,DOY,INTEGER,0,None,0
1,1,T2M,REAL,0,None,0
2,2,RH2M,REAL,0,None,0
3,3,PRECTOTCORR,REAL,0,None,0
4,4,WS2M,REAL,0,None,0
5,5,PS,REAL,0,None,0
6,6,date,TEXT,0,None,0


In [42]:
#joining the two data sets together for further analysis 

query = """
SELECT 
   T2M,
   RH2M,
   PRECTOTCORR,
   WS2M,
   p.avg_pm2_5,
   w.date
FROM Beijing_Weather w
INNER JOIN beijing_pm25_monthly p
    ON strftime('%Y-%m', w.date) = p.month
ORDER BY w.date
"""
df = pd.read_sql(query, connection)
df.head()

,T2M,RH2M,PRECTOTCORR,WS2M,avg_pm2_5,date
0,-7.51,49.30,0.00,0.97,104.984052,2024-02-01
1,-6.63,59.39,0.00,1.02,104.984052,2024-02-02
2,-3.66,55.85,0.00,1.07,104.984052,2024-02-03
3,-4.91,64.77,0.24,1.10,104.984052,2024-02-04
4,-2.74,65.55,0.42,1.15,104.984052,2024-02-05


This shows the combination of the relevant columns in both tables. We are keeping the average monthly pm2_5 because it is one of the most common pollutants to measure air pollution. We have dropped pressure from the weather conditions since it does not fluctuate that often from day to day or month to month. Although they do not share column names, because they share a similar data type, the query is able to run by joining on this modification of date. 

In [44]:
#descending join query 

query = """
SELECT 
    w.date,
    w.T2M AS temperature,
    w.RH2M AS humidity,
    w.PRECTOTCORR AS precipitation,
    w.WS2M AS wind_speed,
    p.avg_pm2_5
FROM Beijing_Weather w
INNER JOIN beijing_pm25_monthly p
    ON strftime('%Y-%m', w.date) = p.month
ORDER BY p.avg_pm2_5 DESC
"""
df = pd.read_sql(query, connection)
df.head(25)

,date,temperature,humidity,precipitation,wind_speed,avg_pm2_5
0,2025-12-01,-2.77,54.63,0.00,2.85,130.243952
1,2025-12-02,-6.83,47.31,0.00,4.06,130.243952
2,2025-12-03,-5.32,56.93,0.00,1.86,130.243952
3,2025-12-04,-3.22,60.84,0.00,2.72,130.243952
4,2025-12-05,-2.01,67.27,0.00,1.50,130.243952
5,2025-12-06,0.58,72.46,0.00,1.77,130.243952
6,2025-12-07,0.51,53.67,0.00,4.31,130.243952
7,2025-12-08,-2.82,61.05,0.00,2.49,130.243952
8,2025-12-09,-1.44,72.05,0.00,1.47,130.243952
9,2025-12-10,-0.68,78.23,0.00,1.47,130.243952


This query allows us to see in what month pm25 is the highest, we can see that it is in December. This tracks with our previous graphs, that shows the winter months are when the air pollution appears to be the worst (even though that was with the median), which could also suggest temperature plays a role in air pollution severity. 

In [46]:
#subquery on months with above the overall pm25 average  
query = """
SELECT *
FROM (
    SELECT 
        w.date,
        w.T2M,
        w.RH2M,
        w.WS2M,
        p.avg_pm2_5
    FROM Beijing_Weather w
    JOIN beijing_pm25_monthly p
        ON strftime('%Y-%m', w.date) = p.month
) t
WHERE avg_pm2_5 > (
    SELECT AVG(avg_pm2_5)
    FROM beijing_pm25_monthly
)
ORDER BY avg_pm2_5 DESC
"""
df = pd.read_sql(query, connection)
df.head()

,date,T2M,RH2M,WS2M,avg_pm2_5
0,2025-12-01,-2.77,54.63,2.85,130.243952
1,2025-12-02,-6.83,47.31,4.06,130.243952
2,2025-12-03,-5.32,56.93,1.86,130.243952
3,2025-12-04,-3.22,60.84,2.72,130.243952
4,2025-12-05,-2.01,67.27,1.50,130.243952


This is another way to show when the pm25 average is relatively high, we are again confirmed that it is typically in December. 

In [47]:
# checking the trend 
query = """
SELECT *
FROM (
    SELECT 
        w.date,
        w.T2M,
        w.RH2M,
        w.WS2M,
        p.avg_pm2_5
    FROM Beijing_Weather w
    JOIN beijing_pm25_monthly p
        ON strftime('%Y-%m', w.date) = p.month
    WHERE w.date BETWEEN '2023-01-01' AND '2024-12-31'
) t
WHERE avg_pm2_5 > (
    SELECT AVG(avg_pm2_5)
    FROM beijing_pm25_monthly
    WHERE month BETWEEN '2023-01' AND '2024-12'
)
ORDER BY avg_pm2_5 DESC
"""
df = pd.read_sql(query, connection)
df.head()

,date,T2M,RH2M,WS2M,avg_pm2_5
0,2024-11-01,12.48,72.30,1.40,115.135139
1,2024-11-02,11.08,83.01,1.53,115.135139
2,2024-11-03,10.29,62.92,3.79,115.135139
3,2024-11-04,4.90,48.58,2.53,115.135139
4,2024-11-05,5.34,56.77,2.13,115.135139


#### Do we see this trend continue in a previous year? 
The query uses a window and interprets the selected data to select the average pm25 data that is the highest, and ranks them accordingly. From this query, we find that no, the average monthly pm25 does not follow the trend in 2025. PM25 was the highest in November in contrast to December in 2025. However, the highest average monthly pm25 amount grew in 2025, which could suggest underlying changes in the year. 

In [49]:
#groupby wind query 

query = """
SELECT 
    CASE
        WHEN w.WS2M < 2 THEN 'Low Wind'
        WHEN w.WS2M BETWEEN 2 AND 5 THEN 'Moderate Wind'
        ELSE 'High Wind'
    END AS wind_category,
    AVG(p.avg_pm2_5) AS avg_pm25
FROM Beijing_Weather w
INNER JOIN beijing_pm25_monthly p
    ON strftime('%Y-%m', w.date) = p.month
GROUP BY wind_category
ORDER BY avg_pm25 DESC
"""
df = pd.read_sql(query, connection)
df.head()

,wind_category,avg_pm25
0,High Wind,84.892873
1,Low Wind,84.146336
2,Moderate Wind,80.329376


#### Does low wind create more pollution?

This query groups the wind speed into low, moderate, and high wind speeds. It appears that no matter the wind conditions, average pm25 is stagant. Oddly, the average pm25 dropped slightly with moderate wind instead of low wind. 